# Project 2 — LLM Inference Profiling
### Roofline decode-latency + KV-concurrency model | vLLM · torch.profiler · AWQ/FP8 · CUDA graphs

---

## How to use this notebook

1. **Runtime → Change runtime type → T4 GPU** (free tier is enough for everything except the FP8 section).
2. Run cells top to bottom. Every section has **theory first, then the experiment that proves it.**
3. Cells marked `[OPTIONAL — HEAVY]` install large packages. Skip them on a first pass.

## The one sentence that this whole project exists to prove

> **Autoregressive decoding is not compute-bound. It is memory-bandwidth-bound. Therefore the way to make it faster is to move fewer bytes — not to do less math.**

Everything below — roofline, KV cache, quantization, batching, CUDA graphs — is a consequence of that sentence.

---

## Section map

| # | Section | What you learn |
|---|---------|----------------|
| 1 | Prefill vs decode | The two phases and why they have opposite bottlenecks |
| 2 | Roofline theory | Arithmetic intensity, the two roofs, where LLM ops sit |
| 3 | Measure your GPU | Empirical peak FLOP/s and HBM bandwidth |
| 4 | Plot the roofline | Your actual hardware ceiling |
| 5 | Parameter & byte accounting | Where every byte of a transformer lives |
| 6 | KV cache math | The formula that governs concurrency |
| 7 | Analytical latency model | Predict TPOT before you run anything |
| 8 | Measure real latency | TTFT, TPOT, and Model Bandwidth Utilization |
| 9 | Batch scaling | Throughput vs latency, and the knee |
| 10 | The 3B-beats-4B result | The concurrency model that produces the 5x |
| 11 | torch.profiler | Kernel-level evidence, launch overhead |
| 12 | CUDA graphs | Killing launch overhead at low batch |
| 13 | Quantization: AWQ / FP8 | Why 4-bit weights cut latency ~4x |
| 14 | vLLM | PagedAttention + continuous batching |
| 15 | Interview cheat sheet | Every question you will be asked |


---
# 1. Prefill vs Decode — the two phases

An LLM inference request has two phases with **completely different performance characteristics.**

### Phase 1 — Prefill (a.k.a. the forward pass over the prompt)

You have a prompt of `S` tokens. You push all `S` tokens through the model **in parallel, in one forward pass.**

- Every matmul is `[S, d] @ [d, d]` — a big, fat, GEMM.
- Per byte of weight you load, you do `S` tokens' worth of math.
- **Arithmetic intensity is high → compute-bound.**
- This determines **TTFT** (Time To First Token).

### Phase 2 — Decode (generating tokens 2, 3, 4, ...)

You generate **one token at a time**, because token `t+1` depends on token `t`. That is the definition of autoregressive.

- Every matmul is `[1, d] @ [d, d]` — a GEMV (matrix-vector). Skinny.
- You load **the entire model's weights from HBM** to produce **one single token**.
- **Arithmetic intensity is terrible → memory-bandwidth-bound.**
- This determines **TPOT** (Time Per Output Token), sometimes called ITL (Inter-Token Latency).

### The consequence

```
Total latency = TTFT + (num_output_tokens - 1) * TPOT
```

For a chat response of 500 tokens, TPOT dominates completely. **So decode is what you optimize.**

### Why this is counterintuitive

People assume "bigger model = more FLOPs = slower". Partly true, but the real reason a 7B model is slower than a 1B model at decode is that you must **stream 14 GB of weights instead of 2 GB** through a fixed-bandwidth pipe, *for every single token*. The GPU's tensor cores sit mostly idle waiting on memory.

> **Interview line:** "During decode the GPU is a very expensive memory controller."


In [ ]:
# --- Environment check -------------------------------------------------------
import torch, platform, sys
print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("cuda avail  :", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("gpu         :", p.name)
    print("VRAM        : %.1f GB" % (p.total_memory / 1e9))
    print("SMs         :", p.multi_processor_count)
    print("compute cap : %d.%d" % (p.major, p.minor))
    # Capability gates you should know:
    #   sm70 (V100)  : fp16 tensor cores
    #   sm75 (T4)    : fp16 + int8 tensor cores  -> AWQ works, FP8 does NOT
    #   sm80 (A100)  : bf16, TF32
    #   sm89 (L4/4090), sm90 (H100) : FP8 tensor cores
else:
    print("!! No GPU. Runtime -> Change runtime type -> T4 GPU")


---
# 2. The Roofline Model

The roofline is a **one-picture performance model**. It answers: *given this kernel, what is the fastest it could possibly run on this hardware, and what is limiting it?*

### Arithmetic Intensity (AI)

$$ \text{AI} = \frac{\text{FLOPs performed}}{\text{Bytes moved from DRAM}} \quad [\text{FLOP/byte}] $$

This is a property of **the algorithm**, not the hardware.

### The two roofs

Attainable performance is the minimum of two ceilings:

$$ P_{\text{attainable}} = \min\big(\underbrace{P_{\text{peak}}}_{\text{compute roof, flat}},\ \underbrace{\text{AI} \times BW_{\text{peak}}}_{\text{memory roof, sloped}}\big) $$

Plotted with log axes: a sloped line (bandwidth-limited) that hits a horizontal line (compute-limited). The corner is the **ridge point**:

$$ \text{AI}_{\text{ridge}} = \frac{P_{\text{peak}}}{BW_{\text{peak}}} $$

- Kernel AI **< ridge** → left of the corner → **memory-bound**. Adding FLOPs is free; reducing bytes is everything.
- Kernel AI **> ridge** → right of the corner → **compute-bound**. Now FLOPs matter.

### Worked example: T4

Peak fp16 tensor core ≈ 65 TFLOP/s, HBM bandwidth ≈ 320 GB/s.

$$ \text{AI}_{\text{ridge}} = \frac{65 \times 10^{12}}{320 \times 10^{9}} \approx 203 \ \text{FLOP/byte} $$

You need to do **~200 floating point operations for every byte you read** just to break even. Now compute the AI of a decode step:

A weight matrix `W` of shape `[d, d]` in fp16 costs `2*d*d` bytes to load. With batch size `B`, the matmul `[B,d] @ [d,d]` costs `2*B*d*d` FLOPs.

$$ \text{AI}_{\text{decode}} = \frac{2Bd^2}{2d^2} = B $$

**The arithmetic intensity of decode is literally just the batch size.** At `B=1`, AI = 1 FLOP/byte, versus a ridge of 203. You are running at roughly **0.5% of peak compute.** That is not a bug — it is the physics of the algorithm.

### And this is the entire justification for batching

Raising `B` walks you rightward along the sloped roof. Each increment of batch is nearly free in latency (you reuse weights you already loaded) but linearly increases throughput — **until AI reaches the ridge point**, after which you are compute-bound and further batching hurts latency for no throughput gain.

> That is why serving engines fight so hard for large effective batch sizes. And what stops you from batching more? **KV cache memory.** Section 6.


---
# 3. Measure your GPU (never trust the datasheet)

Marketing numbers assume perfect conditions. We measure **achievable** peak, which is what the roofline should actually use.

### Method
- **Peak FLOP/s**: large square fp16 matmuls. FLOPs for `[n,n]@[n,n]` = `2*n^3`. Sweep `n`, take the max.
- **Peak bandwidth**: a large device-to-device copy. Bytes = `2 * numel * itemsize` (one read + one write).

### Two things that ruin GPU benchmarks (and that interviewers check you know)

1. **CUDA is asynchronous.** `t=time(); kernel(); print(time()-t)` measures *launch* time, not *execution* time. You must `torch.cuda.synchronize()` or use `torch.cuda.Event`.
2. **First run is a lie.** cuBLAS autotunes and allocates workspace on the first call; the allocator caches. Always do **warmup iterations** and discard them.


In [ ]:
import torch, time
import numpy as np

assert torch.cuda.is_available(), "Enable the GPU runtime."
dev = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True

def timed(fn, warmup=5, iters=20):
    """Correctly time a GPU op with CUDA events. Returns median seconds."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    ts = []
    for _ in range(iters):
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        fn()
        e.record()
        torch.cuda.synchronize()
        ts.append(s.elapsed_time(e) / 1000.0)   # ms -> s
    return float(np.median(ts))                 # median, not mean: robust to outliers

# ---- 1. Peak achievable FLOP/s (fp16 tensor cores) --------------------------
print("Measuring peak compute...")
best_flops = 0.0
for n in [1024, 2048, 4096, 8192]:
    try:
        a = torch.randn(n, n, device=dev, dtype=torch.float16)
        b = torch.randn(n, n, device=dev, dtype=torch.float16)
        t = timed(lambda: torch.matmul(a, b))
        flops = 2.0 * n**3 / t
        best_flops = max(best_flops, flops)
        print("  n=%5d  %6.2f ms  ->  %6.1f TFLOP/s" % (n, t*1e3, flops/1e12))
        del a, b
        torch.cuda.empty_cache()
    except RuntimeError as ex:
        print("  n=%5d  skipped (%s)" % (n, str(ex)[:40]))

# ---- 2. Peak achievable HBM bandwidth ---------------------------------------
print("\nMeasuring peak memory bandwidth...")
best_bw = 0.0
for mb in [64, 256, 512]:
    numel = mb * 1024 * 1024 // 2          # fp16 = 2 bytes
    try:
        src = torch.empty(numel, device=dev, dtype=torch.float16)
        dst = torch.empty(numel, device=dev, dtype=torch.float16)
        t = timed(lambda: dst.copy_(src))
        bytes_moved = 2 * numel * 2        # read src + write dst
        bw = bytes_moved / t
        best_bw = max(best_bw, bw)
        print("  %4d MB  %6.3f ms  ->  %6.1f GB/s" % (mb, t*1e3, bw/1e9))
        del src, dst
        torch.cuda.empty_cache()
    except RuntimeError:
        pass

PEAK_FLOPS = best_flops
PEAK_BW    = best_bw
RIDGE      = PEAK_FLOPS / PEAK_BW

print("\n" + "="*58)
print("PEAK_FLOPS  = %.1f TFLOP/s" % (PEAK_FLOPS/1e12))
print("PEAK_BW     = %.1f GB/s"    % (PEAK_BW/1e9))
print("RIDGE POINT = %.1f FLOP/byte" % RIDGE)
print("="*58)
print("\nInterpretation: any kernel with arithmetic intensity below %.0f" % RIDGE)
print("FLOP/byte CANNOT use the tensor cores fully, no matter how well written.")


---
# 4. Draw your roofline and place the LLM operations on it

Now we plot the ceiling and mark where real LLM kernels live. Watch how far left decode sits.


In [ ]:
import matplotlib.pyplot as plt

ai = np.logspace(-1, 4, 400)
attainable = np.minimum(PEAK_FLOPS, ai * PEAK_BW)

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(ai, attainable / 1e12, lw=2.5, color="black", label="Roofline (attainable)")
ax.axvline(RIDGE, ls="--", color="gray", alpha=0.7)
ax.text(RIDGE*1.1, PEAK_FLOPS/1e12*0.12, "ridge = %.0f FLOP/byte" % RIDGE,
        rotation=90, fontsize=9, color="gray")

# Where real operations sit. For a weight matmul, AI == effective batch size
# (tokens processed per weight load).
points = [
    ("decode B=1",    1,    "tab:red"),
    ("decode B=8",    8,    "tab:orange"),
    ("decode B=64",   64,   "tab:olive"),
    ("decode B=256",  256,  "tab:green"),
    ("prefill S=512", 512,  "tab:blue"),
    ("prefill S=4096",4096, "tab:purple"),
]
for name, x, c in points:
    y = min(PEAK_FLOPS, x * PEAK_BW) / 1e12
    ax.plot(x, y, "o", ms=11, color=c)
    ax.annotate(name, (x, y), textcoords="offset points", xytext=(8, -14), fontsize=9, color=c)

ax.set_xlabel("Arithmetic intensity (FLOP / byte)  ~=  tokens per weight load")
ax.set_ylabel("Attainable performance (TFLOP/s)")
ax.set_title("Roofline: LLM decode lives on the memory-bound slope")
ax.grid(True, which="both", alpha=0.25)
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

eff = (1 * PEAK_BW) / PEAK_FLOPS * 100
print("At batch size 1, decode achieves ~%.2f%% of the GPU's peak compute." % eff)
print("You are paying for tensor cores you are not using. That is normal and expected.")


---
# 5. Parameter and byte accounting for a transformer

To predict latency you must know exactly how many bytes get streamed per token. Let:

- `L` = number of layers
- `d` = hidden size (`d_model`)
- `H` = number of attention heads, `H_kv` = number of **key/value** heads (GQA), `h = d / H` = head dim
- `I` = MLP intermediate size
- `V` = vocab size

### Per layer

**Attention block** (with Grouped Query Attention):
- `W_q`: `d × d`
- `W_k`: `d × (H_kv · h)`
- `W_v`: `d × (H_kv · h)`
- `W_o`: `d × d`

**MLP block**. Modern models (Llama, Qwen, Mistral) use **SwiGLU**, which has *three* matrices, not two:
- `W_gate`: `d × I`, `W_up`: `d × I`, `W_down`: `I × d`

$$ P_{\text{layer}} = \underbrace{2d^2 + 2 d H_{kv} h}_{\text{attention}} + \underbrace{3 d I}_{\text{SwiGLU MLP}} $$

### Total

$$ P_{\text{total}} = L \cdot P_{\text{layer}} + \underbrace{V d}_{\text{embedding}} + \underbrace{V d}_{\text{lm head, if untied}} $$

### Bytes

$$ \text{Bytes} = P_{\text{total}} \times \text{bytes per param} $$

| Precision | bytes/param | 7B model |
|---|---|---|
| fp32 | 4 | 28 GB |
| fp16 / bf16 | 2 | 14 GB |
| int8 / FP8 | 1 | 7 GB |
| int4 (AWQ/GPTQ) | 0.5 | 3.5 GB |

> **The whole point:** at decode, that entire number gets read from HBM **once per token**. Halving the precision halves the time per token. This is why quantization is a *latency* optimization for decode, not just a memory-fitting trick.

### Notes for interviews
- **Why SwiGLU has 3 matrices:** `down(silu(gate(x)) * up(x))`. The gating gives better quality per parameter, at the cost of an extra projection. Typically `I ≈ 2.7·d` instead of `4·d` to keep parameter count comparable.
- **Tied embeddings:** small models often share the input embedding with the output head, saving `V·d` params. Qwen2.5-0.5B does this.


In [ ]:
from dataclasses import dataclass

@dataclass
class ModelSpec:
    name: str
    L: int          # layers
    d: int          # hidden size
    H: int          # attention heads
    H_kv: int       # kv heads (GQA); == H means MHA
    I: int          # mlp intermediate
    V: int          # vocab
    tied: bool = False

    @property
    def head_dim(self): return self.d // self.H

    def params(self):
        h = self.head_dim
        attn = 2*self.d*self.d + 2*self.d*self.H_kv*h
        mlp  = 3*self.d*self.I                 # SwiGLU
        per_layer = attn + mlp
        emb  = self.V*self.d * (1 if self.tied else 2)
        return {
            "attn_per_layer": attn,
            "mlp_per_layer": mlp,
            "per_layer": per_layer,
            "all_layers": per_layer*self.L,
            "embeddings": emb,
            "total": per_layer*self.L + emb,
        }

    def weight_bytes(self, bits=16):
        return self.params()["total"] * bits/8

    def kv_bytes_per_token(self, bits=16):
        # 2 (K and V) * layers * kv_heads * head_dim * bytes
        return 2 * self.L * self.H_kv * self.head_dim * (bits/8)

MODELS = {
    "qwen2.5-0.5b": ModelSpec("Qwen2.5-0.5B", L=24, d=896,  H=14, H_kv=2, I=4864,  V=151936, tied=True),
    "qwen2.5-1.5b": ModelSpec("Qwen2.5-1.5B", L=28, d=1536, H=12, H_kv=2, I=8960,  V=151936, tied=True),
    "qwen2.5-3b":   ModelSpec("Qwen2.5-3B",   L=36, d=2048, H=16, H_kv=2, I=11008, V=151936, tied=True),
    "llama3.2-1b":  ModelSpec("Llama3.2-1B",  L=16, d=2048, H=32, H_kv=8, I=8192,  V=128256, tied=True),
    "llama3.2-3b":  ModelSpec("Llama3.2-3B",  L=28, d=3072, H=24, H_kv=8, I=8192,  V=128256, tied=True),
    "llama3-8b":    ModelSpec("Llama3-8B",    L=32, d=4096, H=32, H_kv=8, I=14336, V=128256, tied=False),
    "mistral-7b":   ModelSpec("Mistral-7B",   L=32, d=4096, H=32, H_kv=8, I=14336, V=32000,  tied=False),
}

print("%-14s %9s %11s %11s %14s" % ("model","params","fp16 GB","int4 GB","KV B/token"))
print("-"*64)
for k, m in MODELS.items():
    p = m.params()["total"]
    print("%-14s %8.2fB %10.2f %11.2f %13.0f" % (
        m.name, p/1e9, m.weight_bytes(16)/1e9, m.weight_bytes(4)/1e9, m.kv_bytes_per_token(16)))

print("\nSanity check: our formula vs the real checkpoint's reported size.")
print("Small differences come from layernorms and rotary buffers, which we ignore (<0.1%).")


---
# 6. KV cache — the formula that governs everything about concurrency

### Why the cache exists

At decode step `t`, attention needs the keys and values of **all previous tokens** `1..t-1`. Those don't change. Recomputing them every step would make generation O(n²) in compute. So you cache them.

$$ \text{KV bytes per token} = 2 \times L \times H_{kv} \times h \times \text{bytes per element} $$

The leading `2` is for **K and V** (not for fp16 — that's the last term). Get this wrong in an interview and you're done.

### Grouped Query Attention (GQA) — the single most important KV optimization

- **MHA** (original): `H_kv = H`. Every query head has its own K/V head.
- **MQA**: `H_kv = 1`. All query heads share one K/V head. Maximum saving, some quality loss.
- **GQA**: `H_kv = H / g` for a small group size. The compromise everyone ships now.

Llama-3-8B: `H=32, H_kv=8` → **4× less KV cache** than MHA, for negligible quality loss. Qwen2.5 is even more aggressive with `H_kv=2`.

> GQA doesn't reduce parameter count much. It reduces **KV cache**, which is what limits how many users you can serve at once.

### The memory budget equation

$$ \text{VRAM} = \underbrace{W}_{\text{weights}} + \underbrace{A}_{\text{activations, framework}} + \underbrace{N \times S \times k}_{\text{KV cache}} $$

Solve for `N`, the number of concurrent sequences:

$$ N_{\max} = \left\lfloor \frac{\text{VRAM} - W - A}{S \times k} \right\rfloor $$

where `k` = KV bytes per token, `S` = sequence length.

**This is the equation behind the "3B outperforms 4B by 5×" claim.** `W` appears with a minus sign in the numerator, so shrinking the model doesn't just cut weight-streaming time linearly — it *frees headroom that multiplies concurrency*, and concurrency is what amortizes the weight load. Two compounding effects. Section 10 quantifies it.


In [ ]:
def concurrency(m: ModelSpec, vram_gb=15.0, seq_len=2048, w_bits=16, kv_bits=16, overhead_gb=1.5):
    """How many concurrent sequences fit? The core serving-capacity model."""
    vram = vram_gb * 1e9
    W = m.weight_bytes(w_bits)
    A = overhead_gb * 1e9          # CUDA context, activations, fragmentation
    free = vram - W - A
    k = m.kv_bytes_per_token(kv_bits)
    per_seq = k * seq_len
    n = max(0, int(free // per_seq))
    return {"weights_GB": W/1e9, "free_for_kv_GB": max(0,free)/1e9,
            "kv_per_seq_MB": per_seq/1e6, "max_concurrent": n}

print("T4 16GB, seq_len=2048, fp16 weights + fp16 KV\n")
print("%-14s %9s %13s %13s %10s" % ("model","wt GB","free KV GB","MB/seq","N_max"))
print("-"*64)
for k_, m in MODELS.items():
    r = concurrency(m)
    print("%-14s %8.2f %12.2f %13.1f %9d" % (m.name, r["weights_GB"], r["free_for_kv_GB"],
                                             r["kv_per_seq_MB"], r["max_concurrent"]))

print("\n--- Effect of quantizing WEIGHTS to int4 (frees room for more KV) ---")
print("%-14s %9s %10s %10s" % ("model","wt GB","N fp16","N int4"))
print("-"*48)
for k_, m in MODELS.items():
    a = concurrency(m, w_bits=16)["max_concurrent"]
    b = concurrency(m, w_bits=4)["max_concurrent"]
    print("%-14s %8.2f %9d %9d" % (m.name, m.weight_bytes(4)/1e9, a, b))


---
# 7. The analytical decode-latency model

Now assemble it. Per decode step, the bytes you must read are:

$$ B_{\text{step}} = \underbrace{W}_{\text{all weights, once}} + \underbrace{N \times S \times k}_{\text{KV cache of every active sequence}} $$

and the time is bounded below by bandwidth:

$$ \boxed{\ \text{TPOT} \ \geq\ \frac{W + N S k}{BW_{\text{peak}}}\ } $$

Throughput in tokens/second across all users:

$$ \text{Throughput} = \frac{N}{\text{TPOT}} = \frac{N \cdot BW}{W + NSk} $$

### Read the limits — this is the whole intuition

- **Small `N`** (`W >> NSk`): `Throughput ≈ N·BW/W`. **Linear in N.** Batching is nearly free — you're re-reading the same weights anyway. This is the regime almost everyone deploys in without realizing it.
- **Large `N`** (`NSk >> W`): `Throughput → BW/(S·k)`. **Saturated.** The KV cache reads now dominate; adding users no longer helps and latency degrades. GQA and KV quantization push this ceiling up by shrinking `k`.

### What "good" looks like: Model Bandwidth Utilization (MBU)

$$ \text{MBU} = \frac{\text{bytes that theoretically must move}}{\text{TPOT} \times BW_{\text{peak}}} $$

- MBU 60–80% → your kernels are efficient; you're near the physical limit. Only quantization or better hardware helps now.
- MBU < 30% at small batch → you are **launch-overhead bound**, not bandwidth bound. Fix with CUDA graphs (Section 12).

MBU is the decode equivalent of MFU (Model FLOPs Utilization) that people quote for training. Knowing to use *MBU* for decode and *MFU* for training/prefill is a strong signal in an interview.


In [ ]:
def predict_tpot(m: ModelSpec, N=1, S=2048, w_bits=16, kv_bits=16, bw=None):
    bw = bw or PEAK_BW
    W = m.weight_bytes(w_bits)
    KV = N * S * m.kv_bytes_per_token(kv_bits)
    return (W + KV) / bw, W, KV

m = MODELS["qwen2.5-1.5b"]
print("Predicted decode latency for %s (S=2048, fp16)\n" % m.name)
print("%5s %11s %11s %11s %14s" % ("N","TPOT ms","tok/s tot","weight%","kv%"))
print("-"*58)
for N in [1,2,4,8,16,32,64,128,256]:
    t, W, KV = predict_tpot(m, N=N)
    print("%5d %10.2f %11.0f %10.0f%% %13.0f%%" % (N, t*1e3, N/t, 100*W/(W+KV), 100*KV/(W+KV)))

print("\nNote where the weight share drops below 50% -- that is the point where")
print("you stop being weight-bound and start being KV-bound. Beyond it, extra")
print("batching buys you much less.")


In [ ]:
# Throughput-vs-batch curves for several models: the saturation picture
fig, ax = plt.subplots(1, 2, figsize=(14,5))
Ns = np.array([1,2,4,8,16,32,64,128,256,512])

for key in ["qwen2.5-0.5b","qwen2.5-1.5b","qwen2.5-3b","llama3-8b"]:
    mm = MODELS[key]
    tp, lat = [], []
    for N in Ns:
        t,_,_ = predict_tpot(mm, N=int(N), S=2048)
        tp.append(N/t); lat.append(t*1e3)
    ax[0].plot(Ns, tp, "o-", label=mm.name)
    ax[1].plot(Ns, lat, "o-", label=mm.name)

ax[0].set_xscale("log", base=2); ax[0].set_yscale("log")
ax[0].set_xlabel("concurrent sequences N"); ax[0].set_ylabel("total tokens/s")
ax[0].set_title("Throughput saturates: linear, then flat"); ax[0].grid(alpha=.3); ax[0].legend()

ax[1].set_xscale("log", base=2)
ax[1].set_xlabel("concurrent sequences N"); ax[1].set_ylabel("TPOT (ms) - per user latency")
ax[1].set_title("Per-user latency: flat, then linear"); ax[1].grid(alpha=.3); ax[1].legend()
plt.tight_layout(); plt.show()

print("The knee is your operating point. Left of it you are wasting the GPU.")
print("Right of it you are degrading every user's latency for no throughput gain.")


---
# 8. Measure it for real

Prediction is worthless without validation. We now load an actual model and measure TTFT and TPOT, then compute MBU to see how close we get to the bandwidth roof.

### Measurement discipline (say these out loud in an interview)
- Warm up (cuBLAS autotune, allocator, kernel JIT).
- `torch.cuda.synchronize()` around every timed region.
- Report **median and p95**, not mean — GPU tails are real.
- Fix the number of generated tokens (`min_new_tokens = max_new_tokens`) so EOS doesn't shorten runs and corrupt the average.
- Use **greedy** decoding to remove sampling variance.


In [ ]:
# Install (Colab usually has recent versions already)
!pip -q install -U "transformers>=4.44" accelerate sentencepiece 2>/dev/null | tail -1
print("ok")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, time, numpy as np, gc

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # swap to 1.5B if you have VRAM headroom

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="cuda", attn_implementation="sdpa"
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
real_bytes = sum(p.numel()*p.element_size() for p in model.parameters())
print("params          : %.3f B" % (n_params/1e9))
print("weight bytes    : %.3f GB" % (real_bytes/1e9))
cfg = model.config
print("layers=%d hidden=%d heads=%d kv_heads=%d inter=%d vocab=%d" % (
    cfg.num_hidden_layers, cfg.hidden_size, cfg.num_attention_heads,
    getattr(cfg,'num_key_value_heads',cfg.num_attention_heads),
    cfg.intermediate_size, cfg.vocab_size))

# Compare our analytic formula against the real checkpoint
spec = ModelSpec("measured", L=cfg.num_hidden_layers, d=cfg.hidden_size,
                 H=cfg.num_attention_heads,
                 H_kv=getattr(cfg,'num_key_value_heads',cfg.num_attention_heads),
                 I=cfg.intermediate_size, V=cfg.vocab_size,
                 tied=getattr(cfg,'tie_word_embeddings',False))
print("\nformula predicts: %.3f B params  (error %.2f%%)" % (
    spec.params()['total']/1e9, 100*abs(spec.params()['total']-n_params)/n_params))
KV_PER_TOKEN = spec.kv_bytes_per_token(16)
print("KV bytes/token  : %.0f  (%.2f MB per 2048-token sequence)" % (KV_PER_TOKEN, KV_PER_TOKEN*2048/1e6))


In [ ]:
@torch.inference_mode()
def bench(prompt_len=256, gen_len=64, batch=1, reps=5):
    """Separately measure TTFT (prefill) and TPOT (decode)."""
    ids = torch.randint(1000, 5000, (batch, prompt_len), device="cuda")
    attn = torch.ones_like(ids)

    # ---- warmup ----
    for _ in range(2):
        model.generate(ids, attention_mask=attn, max_new_tokens=4,
                       min_new_tokens=4, do_sample=False,
                       pad_token_id=tok.eos_token_id)
    torch.cuda.synchronize()

    # ---- TTFT: prefill only = generate exactly 1 token ----
    ttfts = []
    for _ in range(reps):
        torch.cuda.synchronize(); t0 = time.perf_counter()
        model.generate(ids, attention_mask=attn, max_new_tokens=1, min_new_tokens=1,
                       do_sample=False, pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize(); ttfts.append(time.perf_counter()-t0)
    ttft = float(np.median(ttfts))

    # ---- full run: prefill + (gen_len-1) decode steps ----
    fulls = []
    for _ in range(reps):
        torch.cuda.synchronize(); t0 = time.perf_counter()
        model.generate(ids, attention_mask=attn, max_new_tokens=gen_len, min_new_tokens=gen_len,
                       do_sample=False, pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize(); fulls.append(time.perf_counter()-t0)
    full = float(np.median(fulls))

    # Isolate decode by subtracting prefill  <- this is the key trick
    tpot = (full - ttft) / (gen_len - 1)

    # MBU: bytes that MUST move per decode step vs what bandwidth allows
    avg_ctx = prompt_len + gen_len/2
    must_move = real_bytes + batch * avg_ctx * KV_PER_TOKEN
    mbu = must_move / (tpot * PEAK_BW)

    return {"batch":batch, "ttft_ms":ttft*1e3, "tpot_ms":tpot*1e3,
            "decode_tok_s": batch/tpot, "mbu_pct": mbu*100,
            "prefill_tok_s": batch*prompt_len/ttft}

r = bench()
print("TTFT              : %.1f ms   (prefill %.0f tok/s)" % (r["ttft_ms"], r["prefill_tok_s"]))
print("TPOT              : %.2f ms   -> %.1f tokens/s for this user" % (r["tpot_ms"], 1000/r["tpot_ms"]))
print("MBU               : %.1f%%" % r["mbu_pct"])
print()
pred,_,_ = predict_tpot(spec, N=1, S=288)
print("Roofline lower bound for TPOT: %.2f ms" % (pred*1e3))
print("We measured                  : %.2f ms  (%.1fx the theoretical floor)" % (r["tpot_ms"], r["tpot_ms"]/(pred*1e3)))
print("""
If MBU is low (<30%), you are NOT bandwidth bound -- you are bound by Python
overhead and kernel launch latency in HF generate(). Sections 11-12 prove it
and fix it. This gap is the single most important finding of the project.""")


---
# 9. Batch scaling — measured, not predicted

Sweep batch size and watch throughput climb while per-token latency stays almost flat. This is the empirical version of the roofline slope.


In [ ]:
rows = []
for B in [1,2,4,8,16,32]:
    try:
        rows.append(bench(prompt_len=256, gen_len=32, batch=B, reps=3))
        print("B=%3d  TPOT %6.2f ms  |  %7.1f tok/s  |  MBU %5.1f%%" %
              (B, rows[-1]["tpot_ms"], rows[-1]["decode_tok_s"], rows[-1]["mbu_pct"]))
        torch.cuda.empty_cache()
    except RuntimeError as e:
        print("B=%3d  OOM" % B); break

Bs   = [r["batch"] for r in rows]
tps  = [r["decode_tok_s"] for r in rows]
lats = [r["tpot_ms"] for r in rows]
mbus = [r["mbu_pct"] for r in rows]

fig, ax = plt.subplots(1,3, figsize=(16,4.5))
ax[0].plot(Bs, tps, "o-", color="tab:green"); ax[0].set_title("Throughput (tok/s)")
ax[1].plot(Bs, lats,"o-", color="tab:red");   ax[1].set_title("TPOT (ms) per user")
ax[2].plot(Bs, mbus,"o-", color="tab:blue");  ax[2].set_title("MBU (%)")
for a in ax: a.set_xlabel("batch size"); a.set_xscale("log", base=2); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print("\nThroughput grew %.1fx while per-user latency grew only %.1fx." %
      (tps[-1]/tps[0], lats[-1]/lats[0]))
print("That gap is free money, and it is the entire economic argument for batched serving.")


---
# 10. The headline result: why a 3B model beats a 4B model by ~5×

This is the number on your resume. Here is how to derive and defend it.

### The naive (wrong) expectation
"4B/3B = 1.33, so the 3B should be ~1.33× faster." That reasoning only accounts for weight streaming at batch size 1.

### The actual mechanism — three compounding effects

**1. Weight streaming (linear, ~1.3×).** Fewer bytes per decode step.

**2. Concurrency headroom (superlinear).** From Section 6:

$$ N_{\max} = \frac{\text{VRAM} - W - A}{S \cdot k} $$

Weights sit in the **numerator with a minus sign**. On a 16 GB card with ~1.5 GB overhead: an 8 GB model leaves 6.5 GB for KV; a 6 GB model leaves 8.5 GB. That's 1.3× less weight but **1.31× more KV room** — and if the smaller model also has fewer layers/KV heads, `k` drops too, multiplying again.

**3. Throughput is `N / TPOT`.** More `N` **and** lower `TPOT` → the ratio compounds:

$$ \frac{\text{TP}_{3B}}{\text{TP}_{4B}} = \frac{N_{3B}}{N_{4B}} \times \frac{\text{TPOT}_{4B}}{\text{TPOT}_{3B}} $$

Roughly 2× from concurrency and 2.5× from batched weight amortization → **~5×**. Add an architectural difference (a 3B with GQA `H_kv=2` vs a 4B with `H_kv=8` is 4× less KV per token) and you can exceed 5× easily.

> **The defensible claim:** "5× *throughput* at fixed VRAM and fixed SLA, not 5× single-stream latency." Always state the operating point. If someone says "5× sounds too high," the answer is: *it's a throughput ratio under a memory constraint, and memory constraints produce superlinear effects.*


In [ ]:
def serving_throughput(m, vram_gb=15.0, S=2048, w_bits=16, kv_bits=16, bw=None):
    bw = bw or PEAK_BW
    c = concurrency(m, vram_gb=vram_gb, seq_len=S, w_bits=w_bits, kv_bits=kv_bits)
    N = max(1, c["max_concurrent"])
    t,_,_ = predict_tpot(m, N=N, S=S, w_bits=w_bits, kv_bits=kv_bits, bw=bw)
    return {"N":N, "tpot_ms":t*1e3, "throughput":N/t, "weights_GB":c["weights_GB"]}

# Construct a matched pair differing only in size, to isolate the effect
small = ModelSpec("3B (GQA kv=2)", L=36, d=2048, H=16, H_kv=2, I=11008, V=151936, tied=True)
big   = ModelSpec("4B (GQA kv=8)", L=36, d=2560, H=20, H_kv=8, I=13824, V=151936, tied=True)

print("%-16s %8s %7s %11s %14s" % ("model","wt GB","N_max","TPOT ms","tok/s total"))
print("-"*62)
res = {}
for m in [small, big]:
    r = serving_throughput(m)
    res[m.name] = r
    print("%-16s %7.2f %7d %10.2f %13.0f" % (m.name, r["weights_GB"], r["N"], r["tpot_ms"], r["throughput"]))

ratio = res[small.name]["throughput"] / res[big.name]["throughput"]
print("\nSPEEDUP = %.2fx" % ratio)
print("  from concurrency : %.2fx" % (res[small.name]["N"]/res[big.name]["N"]))
print("  from per-step lat: %.2fx" % (res[big.name]["tpot_ms"]/res[small.name]["tpot_ms"]))
print("\nDecomposing the number like this is what makes the claim credible.")


In [ ]:
# Sensitivity: does the result survive different VRAM budgets and context lengths?
fig, ax = plt.subplots(1,2, figsize=(14,4.5))

vrams = [12,16,24,40,80]
ax[0].plot(vrams, [serving_throughput(small, vram_gb=v-1)["throughput"]/
                   serving_throughput(big,   vram_gb=v-1)["throughput"] for v in vrams], "o-")
ax[0].set_xlabel("GPU VRAM (GB)"); ax[0].set_ylabel("3B / 4B throughput ratio")
ax[0].set_title("Advantage shrinks as VRAM grows"); ax[0].grid(alpha=.3)
ax[0].axhline(1.0, ls="--", color="gray")

seqs = [512,1024,2048,4096,8192]
ax[1].plot(seqs, [serving_throughput(small, S=s)["throughput"]/
                  serving_throughput(big,   S=s)["throughput"] for s in seqs], "o-", color="tab:purple")
ax[1].set_xscale("log", base=2); ax[1].set_xlabel("context length")
ax[1].set_title("Advantage grows with longer context"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print("""KEY INSIGHT to state in an interview:
The small-model advantage is a MEMORY-CONSTRAINT phenomenon. On an 80GB H100
with short contexts it mostly evaporates. On a 16GB T4 with 8K context it is
enormous. 'Which model is faster' is not a property of the model -- it is a
property of the (model, hardware, context length, SLA) tuple.""")


---
# 11. torch.profiler — get the kernel-level evidence

The roofline tells you the ceiling. The profiler tells you why you're below it.

### What to look for

| Symptom | Diagnosis | Fix |
|---|---|---|
| Sum of kernel times << wall time | **Gaps** — CPU can't launch fast enough | CUDA graphs, `torch.compile` |
| Top kernel is `elementwise`/`copy` | Unfused ops | compile / fused kernels |
| Top kernel is GEMM and MBU is high | Genuinely bandwidth bound | quantize |
| Many tiny kernels (<10µs each) | Launch overhead dominates | CUDA graphs |

### The critical number: **kernel launch overhead is ~5–10 µs**

A 0.5B model has 24 layers × ~10 kernels = ~240 kernel launches per token. At 7 µs each that's **~1.7 ms of pure launch overhead per token** — which can easily exceed the actual compute time at batch size 1. That is exactly why small models feel slower than the roofline predicts.


In [ ]:
from torch.profiler import profile, ProfilerActivity, record_function

ids = torch.randint(1000,5000,(1,128), device="cuda")
attn = torch.ones_like(ids)

with torch.inference_mode():
    for _ in range(3):   # warmup
        model.generate(ids, attention_mask=attn, max_new_tokens=8, min_new_tokens=8,
                       do_sample=False, pad_token_id=tok.eos_token_id)
    torch.cuda.synchronize()

    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
                 record_shapes=True) as prof:
        with record_function("decode_16_tokens"):
            model.generate(ids, attention_mask=attn, max_new_tokens=16, min_new_tokens=16,
                           do_sample=False, pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize()

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=18))


In [ ]:
# The single most revealing computation in the whole notebook:
# how much of the wall clock is the GPU actually busy?
evts = prof.key_averages()
gpu_us = sum(e.self_device_time_total for e in evts if e.self_device_time_total > 0)

torch.cuda.synchronize(); t0 = time.perf_counter()
with torch.inference_mode():
    model.generate(ids, attention_mask=attn, max_new_tokens=16, min_new_tokens=16,
                   do_sample=False, pad_token_id=tok.eos_token_id)
torch.cuda.synchronize(); wall_us = (time.perf_counter()-t0)*1e6

n_launches = sum(e.count for e in evts if e.self_device_time_total > 0)

print("wall clock        : %8.0f us" % wall_us)
print("GPU busy          : %8.0f us" % gpu_us)
print("IDLE (gaps)       : %8.0f us   <-- CPU launch overhead + Python" % (wall_us-gpu_us))
print("GPU utilization   : %8.1f %%"  % (100*gpu_us/wall_us))
print("kernel launches   : %8d  (%.0f per token)" % (n_launches, n_launches/16))
print("""
If utilization is well under 50%, the GPU spends most of its time WAITING for
Python to tell it what to do next. No amount of quantization fixes that.
The fix is to stop launching kernels one at a time -> CUDA graphs.""")


---
# 12. CUDA graphs — eliminate launch overhead

### The problem
Normal execution: CPU launches kernel → GPU runs it → CPU launches next → ... Each launch costs ~5–10 µs of CPU-side work. At batch 1 the kernels themselves may only take 20 µs. You're spending 30% of your time on bookkeeping.

### The fix
A **CUDA graph** records the entire sequence of kernel launches once, as a DAG, then replays the whole thing with **a single CPU call**. Hundreds of launches become one.

### The constraint that everything else follows from
> **A graph replays the exact same kernels on the exact same memory addresses.**

So the computation must be **static**: fixed shapes, fixed pointers, no data-dependent control flow. But a KV cache *grows* every step, which changes shapes. Contradiction.

### The resolution: StaticCache
Pre-allocate the KV cache at `max_length` and **write into it at an index** instead of concatenating. Shapes are then constant; only the contents and a position counter change. This is why `StaticCache` exists in HuggingFace, and it is a prerequisite for `torch.compile(mode="reduce-overhead")` — which is HF's way of saying "use CUDA graphs."

### Trade-offs to mention
- Memory: you pay for `max_length` KV up front, even for short sequences.
- Compile time: 1–3 minutes of graph capture and recompilation per distinct shape.
- Recompiles on shape change — hence `mark_static` / bucketing in production.
- **Benefit is largest at small batch and small models**, i.e. exactly the launch-bound regime. At batch 256 it barely matters.


In [ ]:
from transformers import StaticCache
import torch, time, numpy as np

def bench_static(compiled=False, gen=32, prompt=128, reps=5):
    mdl = model
    ids = torch.randint(1000,5000,(1,prompt), device="cuda")
    attn = torch.ones_like(ids)
    if compiled:
        mdl.generation_config.cache_implementation = "static"
        mdl.forward = torch.compile(mdl.forward, mode="reduce-overhead", fullgraph=True)
    ts=[]
    with torch.inference_mode():
        for _ in range(3):   # warmup: also triggers compile + graph capture
            mdl.generate(ids, attention_mask=attn, max_new_tokens=gen, min_new_tokens=gen,
                         do_sample=False, pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize()
        for _ in range(reps):
            t0=time.perf_counter()
            mdl.generate(ids, attention_mask=attn, max_new_tokens=gen, min_new_tokens=gen,
                         do_sample=False, pad_token_id=tok.eos_token_id)
            torch.cuda.synchronize(); ts.append(time.perf_counter()-t0)
    return float(np.median(ts))

base = bench_static(compiled=False)
print("eager                     : %.1f ms  (%.2f ms/token)" % (base*1e3, base*1e3/32))

print("\nCompiling (this takes 1-3 minutes -- graph capture)...")
try:
    comp = bench_static(compiled=True)
    print("compiled + CUDA graphs    : %.1f ms  (%.2f ms/token)" % (comp*1e3, comp*1e3/32))
    print("SPEEDUP                   : %.2fx" % (base/comp))
    print("\nThis speedup came from launching FEWER TIMES, not computing less.")
    print("Total FLOPs and total bytes moved are identical. Only the CPU-side")
    print("overhead changed. That is the definition of an overhead-bound workload.")
except Exception as e:
    print("compile failed on this runtime:", str(e)[:200])
    print("(Common on older torch/transformers combos. The concept still stands.)")


---
# 13. Quantization — AWQ and FP8

Since decode time ≈ `weight_bytes / bandwidth`, **cutting bits cuts latency proportionally.** 4-bit weights ≈ 4× faster decode at batch 1. Quantization is a *speed* technique here, not just a memory one.

### AWQ — Activation-aware Weight Quantization

The insight that makes AWQ good: **not all weights matter equally, and importance is revealed by activations, not by weight magnitude.**

1. Run calibration data, observe the **activation** magnitude per input channel.
2. Identify the ~1% of channels with large activations — these are the *salient* ones. Quantization error there gets amplified into the output.
3. Instead of keeping them in fp16 (which makes ugly mixed-precision kernels), **scale them up before quantizing** by a per-channel factor `s`, and fold `1/s` into the previous layer. Larger values use more of the quantization grid → relatively less error.
4. Everything is uniformly 4-bit → clean, fast kernels.

**vs GPTQ:** GPTQ minimizes layer-wise reconstruction error using second-order (Hessian) information, layer by layer. It's more expensive to produce and can overfit the calibration set. AWQ needs no backprop, generalizes better, and usually has faster kernels. **vs bitsandbytes NF4:** convenient and training-friendly (used in QLoRA — see Project 3) but its inference kernels are slower than AWQ's.

**Group size (`g=128`):** one scale/zero-point per 128 weights. Smaller group = better accuracy, more metadata overhead. Effective bits ≈ `4 + 16/g + 4/g`.

### FP8 (E4M3 / E5M2)

8-bit **floating point**, native on Hopper (H100, sm90) and Ada (L4/4090, sm89). **Not on T4.**

- `E4M3` (4 exponent, 3 mantissa bits): more precision, less range → **weights and activations**.
- `E5M2` (5 exponent, 2 mantissa): more range, less precision → **gradients**.

Why FP8 beats INT8: floating point keeps **dynamic range**, so outlier activations don't wreck the scale. INT8 activation quantization needs elaborate outlier handling (that's what LLM.int8() and SmoothQuant are for). FP8 largely sidesteps it. And unlike 4-bit, FP8 also speeds up **compute** because the tensor cores run FP8 natively at 2× the FP16 rate.

### When 4-bit does NOT help
At large batch, you're compute-bound, and 4-bit weights must be **dequantized to fp16 to do the matmul** — that dequant is extra work. So AWQ can be *slower* than fp16 at very large batch. Another "it depends on the operating point" answer that scores points.


In [ ]:
# Analytical impact of quantization on decode latency and concurrency
print("Effect of weight precision (Qwen2.5-3B, S=2048, single stream)\n")
m3 = MODELS["qwen2.5-3b"]
base_t = None
print("%-12s %10s %11s %10s %10s" % ("precision","wt GB","TPOT ms","speedup","N_max"))
print("-"*58)
for bits, name in [(16,"fp16"),(8,"int8/fp8"),(4,"int4 AWQ")]:
    t,_,_ = predict_tpot(m3, N=1, S=2048, w_bits=bits)
    base_t = base_t or t
    N = concurrency(m3, w_bits=bits)["max_concurrent"]
    print("%-12s %9.2f %10.2f %9.2fx %9d" % (name, m3.weight_bytes(bits)/1e9, t*1e3, base_t/t, N))

print("\nAlso quantize the KV CACHE (fp8 KV) -- often forgotten, big win at long context:")
print("%-12s %10s %10s" % ("kv bits","MB/seq@8k","N_max"))
print("-"*34)
for kb in [16,8]:
    c = concurrency(m3, seq_len=8192, w_bits=4, kv_bits=kb)
    print("%-12s %9.1f %9d" % (kb, c["kv_per_seq_MB"], c["max_concurrent"]))


In [ ]:
# [OPTIONAL - HEAVY] Run a real AWQ model. Works on T4 (sm75).
RUN_AWQ = False   # flip to True

if RUN_AWQ:
    !pip -q install autoawq
    from awq import AutoAWQForCausalLM
    from transformers import AutoTokenizer
    q_id = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
    qtok = AutoTokenizer.from_pretrained(q_id)
    qmodel = AutoAWQForCausalLM.from_quantized(q_id, fuse_layers=True, device_map="cuda")
    ids = torch.randint(1000,5000,(1,128), device="cuda")
    import time, numpy as np
    ts=[]
    with torch.inference_mode():
        for _ in range(3): qmodel.generate(ids, max_new_tokens=32, do_sample=False)
        torch.cuda.synchronize()
        for _ in range(5):
            t0=time.perf_counter()
            qmodel.generate(ids, max_new_tokens=32, min_new_tokens=32, do_sample=False)
            torch.cuda.synchronize(); ts.append(time.perf_counter()-t0)
    print("AWQ int4 TPOT: %.2f ms/token" % (np.median(ts)*1e3/32))
else:
    print("Set RUN_AWQ=True to benchmark a real 4-bit model.")
    print("Expect roughly 2-3x faster decode than fp16 at batch 1 (not the full 4x,")
    print("because dequantization costs something and launch overhead does not shrink).")


---
# 14. vLLM — PagedAttention and continuous batching

Two ideas. Both are systems ideas, not ML ideas — which is why they were such a large win.

### Problem 1: KV cache memory fragmentation

Naive serving pre-allocates a contiguous KV buffer of `max_seq_len` per request. If a user generates 50 tokens out of a 2048 budget, **97.5% of that allocation is wasted**. Published measurements put naive-system utilization around 20–40%.

### PagedAttention — apply OS virtual memory to the KV cache

- Split the cache into fixed-size **blocks** (e.g. 16 tokens each).
- Each sequence gets a **block table** mapping logical positions → physical blocks, exactly like a page table.
- Blocks are allocated **on demand** as generation proceeds. No contiguity required → **near-zero fragmentation** (only the last partial block, <4% waste).
- **Copy-on-write block sharing:** parallel samples or a shared system prompt reference the same physical blocks until one diverges. Beam search and `n>1` sampling become cheap.

Utilization goes from ~30% to ~96%, so you fit **2–4× more concurrent sequences** in the same VRAM — and by Section 7, more `N` means more throughput.

### Problem 2: static batching wastes the tail

With static batching, a batch of 8 requests finishes when the **longest** one finishes. If seven need 20 tokens and one needs 500, seven GPU slots idle for 480 steps.

### Continuous batching (a.k.a. iteration-level scheduling)
The scheduler runs **per decode step**, not per request. Finished sequences are evicted immediately; queued ones join mid-flight. The batch is refilled continuously. Reported gains: up to ~23× throughput vs naive HF serving, mostly from these two ideas together.

### Other things vLLM does that are worth naming
- **Chunked prefill** — split long prefills into chunks and interleave with decode, so one long prompt doesn't stall everyone's token stream (fixes TPOT spikes).
- **Prefix caching** — hash and reuse KV blocks for a shared system prompt across requests.
- **Speculative decoding** — a small draft model proposes `k` tokens, the big model verifies all `k` in **one** forward pass. Since decode is memory-bound, verifying 5 tokens costs almost the same as generating 1. Free 2–3× if the draft accepts often. **This is the purest possible exploitation of the roofline.**
- **Guided/structured decoding** via FSM-constrained logit masking.


In [ ]:
# [OPTIONAL - HEAVY] vLLM. ~5 min install, and Colab often needs a restart after.
RUN_VLLM = False

if RUN_VLLM:
    !pip -q install vllm
    from vllm import LLM, SamplingParams
    llm = LLM(model="Qwen/Qwen2.5-0.5B-Instruct", dtype="half",
              gpu_memory_utilization=0.85, max_model_len=2048,
              enable_prefix_caching=True)
    prompts = ["Explain the roofline model in one sentence."]*64
    sp = SamplingParams(temperature=0.0, max_tokens=64)
    import time
    t0=time.perf_counter(); outs = llm.generate(prompts, sp); dt=time.perf_counter()-t0
    ntok = sum(len(o.outputs[0].token_ids) for o in outs)
    print("64 requests in %.2fs -> %.0f tok/s aggregate" % (dt, ntok/dt))
    print("Compare against the HF number from Section 9 at the same batch size.")
else:
    print("""Set RUN_VLLM=True to run it. What you should observe and be able to explain:

  * vLLM aggregate throughput is typically 5-20x HF generate() at high concurrency.
  * The gain comes from (a) continuous batching keeping the GPU always full,
    (b) PagedAttention fitting more sequences, (c) fused CUDA kernels and
    CUDA graphs removing launch overhead.
  * At batch size 1 the gap is much smaller -- vLLM's advantage is a
    THROUGHPUT advantage, not a single-stream latency advantage.""")


---
# 15. Consolidated results + interview cheat sheet

Run the cell below to print a summary of everything you measured. Then read the Q&A.


In [ ]:
print("="*70)
print("LLM INFERENCE PROFILING - RESULTS SUMMARY")
print("="*70)
print("Hardware")
print("  GPU                 : %s" % torch.cuda.get_device_properties(0).name)
print("  Measured peak       : %.1f TFLOP/s, %.0f GB/s" % (PEAK_FLOPS/1e12, PEAK_BW/1e9))
print("  Ridge point         : %.0f FLOP/byte" % RIDGE)
print("\nModel: %s" % MODEL_ID)
print("  Params / weights    : %.2fB / %.2f GB fp16" % (n_params/1e9, real_bytes/1e9))
print("  KV per token        : %.0f bytes (%.2f MB @ 2048 ctx)" % (KV_PER_TOKEN, KV_PER_TOKEN*2048/1e6))
print("\nMeasured (batch=1, prompt=256, gen=64)")
print("  TTFT                : %.1f ms" % r["ttft_ms"])
print("  TPOT                : %.2f ms" % r["tpot_ms"])
print("  MBU                 : %.1f%%" % r["mbu_pct"])
print("\nConclusions")
print("  1. Decode AI = batch size; at B=1 that is %.0fx below the ridge." % RIDGE)
print("  2. Throughput scales ~linearly with batch until KV reads dominate.")
print("  3. Small-model advantage is superlinear because weights sit in the")
print("     numerator of the concurrency equation.")
print("  4. At small batch the binding constraint is often kernel LAUNCH")
print("     overhead, not bandwidth -- CUDA graphs, not quantization.")
print("="*70)


---
## Interview Q&A — the questions you will actually get

**Q: Why is decode memory-bound but prefill compute-bound?**
Prefill processes `S` tokens per weight load (AI ≈ `S`); decode processes `B` tokens per weight load (AI ≈ `B`). With `B=1` and a ridge point around 200 FLOP/byte, decode runs at well under 1% of peak compute.

**Q: Derive TPOT.**
`TPOT ≥ (W + N·S·k) / BW`, where `W` = weight bytes, `k` = `2·L·H_kv·h·bytes` KV per token. The `2` is K and V.

**Q: Why does batching improve throughput without hurting latency much?**
Because you're re-reading the same weights regardless. Extra sequences ride along on bytes already in flight. It stops working once `N·S·k` exceeds `W`.

**Q: How does GQA help? Does it reduce FLOPs?**
Barely reduces FLOPs. It reduces **KV cache size** by `H/H_kv`, which raises max concurrency, which raises throughput. It's a memory optimization dressed as an architecture change.

**Q: Your 3B beat your 4B by 5× — that seems too high.**
It's a *throughput at fixed VRAM* ratio, not single-stream latency. Weights appear in the numerator of `N_max = (VRAM − W − A)/(S·k)`, so shrinking the model both cuts per-step bytes and multiplies concurrency. Those compound. The advantage shrinks on an 80 GB card and grows with context length.

**Q: When would you NOT quantize to 4-bit?**
When you're compute-bound (large batch, or prefill-heavy traffic) — dequantization becomes pure overhead. Also when the accuracy loss violates your eval gate, which you should be measuring (see Project 3).

**Q: How do CUDA graphs speed things up if the FLOPs are identical?**
They don't touch the GPU work at all. They remove **CPU-side kernel launch overhead** (~5–10 µs × hundreds of kernels per token). It's a fix for an overhead-bound regime, which you diagnose by finding GPU utilization well below 100% in the profiler.

**Q: Why does StaticCache exist?**
CUDA graphs require static shapes and static memory addresses. A growing KV cache violates both. StaticCache pre-allocates to `max_length` and writes at an index, keeping shapes constant.

**Q: What is PagedAttention in one sentence?**
Virtual memory for the KV cache: fixed-size blocks plus a per-sequence block table, giving on-demand allocation, near-zero fragmentation, and copy-on-write sharing of common prefixes.

**Q: Why is speculative decoding nearly free?**
Because decode is memory-bound. Verifying `k` draft tokens in one forward pass costs about the same as generating one token — you're loading the same weights either way. You convert unused compute into tokens.

**Q: How do you know your kernels are good?**
MBU for decode, MFU for prefill/training. MBU 60–80% means near the bandwidth roof. Below 30% at small batch means launch-overhead bound.

**Q: How would you set up a latency SLA?**
Separate TTFT and TPOT budgets, measure p50/p95/p99 not mean, choose batch size at the knee where throughput plateaus, and use chunked prefill so long prompts don't spike other users' TPOT.

---

## What to say this project *is*, in one breath

> "I built an analytical roofline model for autoregressive decode — TPOT bounded by `(W + N·S·k)/BW` — validated it against measured TTFT/TPOT with torch.profiler, and used it to explain a 5× throughput advantage of a 3B over a 4B model as a concurrency effect rather than a parameter-count effect. Then I closed the gap between predicted and measured latency using CUDA graphs for the launch-bound regime and AWQ for the bandwidth-bound regime."
